# Rossmann Store Sales — Initial EDA

Loads and cleans the raw Kaggle data via `src/data_prep.py`, then looks at overall shape and quality issues found during initial inspection.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
from data_prep import load_raw, clean_store, fill_test_open, build_train_store

pd.set_option('display.max_columns', None)

In [2]:
train, test, store = load_raw()

store_clean = clean_store(store)
test_clean = fill_test_open(test)
train_store = build_train_store(train, store_clean)

train_store.shape, test_clean.shape, store_clean.shape

((1017209, 20), (41088, 8), (1115, 12))

In [3]:
train_store.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,CompetitionDistanceUnknown,CompetitionOpenUnknown
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,,0,0
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",0,0
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",0,0
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,0.0,0.0,,0,0
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,0.0,0.0,,0,0


In [4]:
test_clean.head()

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0
3,4,8,4,2015-09-17,1.0,1,0,0
4,5,9,4,2015-09-17,1.0,1,0,0


## Initial inspection summary (2026-08-03)

**Row counts**
- `train.csv`: 1,017,209 rows x 9 cols, all 1,115 stores, daily records Jan 1 2013 - Jul 31 2015.
- `test.csv`: 41,088 rows x 8 cols, Aug 1 - Sep 17 2015 (immediately after train ends, no overlap). Only 856 of the 1,115 stores appear.
- `store.csv`: 1,115 rows x 10 cols, one row per store.
- `sample_submission.csv`: 41,088 rows x 2 cols, matches test.csv 1:1 on `Id`.

**Closed-store / zero-sales pattern**
- 172,871 rows in train have `Sales == 0`.
- 172,817 of those are explained by `Open == 0` (the store was closed that day) — closed stores reliably report 0 sales.

**The 54 anomaly rows**
- The remaining 54 rows have `Open == 1` but `Sales == 0` — i.e. the store was marked open yet recorded no sales.
- Spread across 41 different stores and dates from 2013 through 2015, with no shared pattern: no state holidays, a mix of promo/non-promo days, a mix of weekdays.
- Likely genuine data-entry quirks rather than a systematic issue. Flagged as an edge case for modeling (e.g. exclude from training or treat as missing) rather than something a rule can clean up.

**Structural nulls in store.csv**
- 544 of 1,115 stores have `Promo2 == 0` (never opted into the continuity promo). For these, `Promo2SinceWeek`, `Promo2SinceYear`, and `PromoInterval` are null — this is *structural* (not applicable), not missing data. `clean_store()` fills these with `0` / `0` / `""` respectively.
- `CompetitionDistance` (3 nulls) and `CompetitionOpenSinceMonth`/`CompetitionOpenSinceYear` (354 nulls each) were investigated separately:
  - The 3 `CompetitionDistance` nulls (stores 291, 622, 879) also have null `CompetitionOpenSinceMonth`/`Year` — these stores have no competition data at all. `clean_store()` imputes `CompetitionDistance` with the median (~2,325m) and flags it via `CompetitionDistanceUnknown`.
  - The other 351 `CompetitionOpenSinceMonth`/`Year` nulls have a known `CompetitionDistance` — a competitor exists at a known distance, just an unrecorded open date, with no skew by `StoreType` or `Promo2`. Left null on purpose (flagged via `CompetitionOpenUnknown`), deferring the "months since competition opened" imputation decision to feature engineering rather than baking in an unverified assumption now.

**The Store 622 issue in test.csv**
- All 11 missing `Open` values in test.csv belong to a single store: Store 622 (rows for dates 2015-09-05 through 2015-09-17).
- `fill_test_open()` fills these with `1` (assumed open), since Store 622 has no other indication of being closed and Rossmann stores are open by default outside of holidays.